# Lesson 10 - उत्पादनमा AI एजेन्टहरू

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरी AI एजेन्टहरूको **उत्पादन ढाँचाहरू** सिक्नुहुनेछ। हामीले समेटेका छौं:

- **अवलोकन क्षमता** — एजेन्ट अन्तरक्रियाहरूमा समय निर्धारण र लगिङ थप्ने
- **मूल्याङ्कन** — प्रतिक्रिया गुणस्तर मापन गर्न मूल्याङ्कन गर्ने एजेन्ट प्रयोग गर्ने
- **खर्च व्यवस्थापन** — टोकन अनुकूलन र मोडल चयनका रणनीतिहरू

परिदृश्य हो एक **यात्रा एजेन्ट** जसले प्रयोगकर्ताहरूलाई यात्रा योजना बनाउन मद्दत गर्छ, र यसको माथि निगरानी र मूल्याङ्कन थपिएको छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचारहरू

AI एजेन्टलाई प्रोटोटाइपबाट उत्पादनमा सार्न तीन मुख्य आधारहरूमा ध्यान दिन आवश्यक छ:

1. **अवलोकनयोग्यता** — तपाईँलाई थाहा हुनुपर्छ एजेन्ट के गर्दैछ, कति समय लाग्छ, र कुन उपकरणहरू कल गर्दछ। ट्रेसिङ र लगिङ बिना उत्पादन समस्याहरू डिबग गर्न लगभग असम्भव हुन्छ।

2. **मूल्यांकन** — स्वचालित गुणस्तर जाँचहरूले सुनिश्चित गर्छ कि एजेन्टका प्रतिक्रियाहरू समयसँगै सही, पूर्ण, र सहायक रहन्छन्। मूल्यांकनकर्ताको एजेन्टले निर्दिष्ट मापदण्डहरूको विरूद्ध प्रतिक्रियाहरूको स्कोर दिन सक्छ।

3. **लागत व्यवस्थापन** — टोकन प्रयोगले लागतमा सिधा प्रभाव पार्छ। प्रॉम्प्ट अनुकूलन, मोडेल चयन, र क्यासिङ जस्ता रणनीतिहरूले गुणस्तर नघटाई खर्चलाई नियन्त्रणमा राख्न मद्दत गर्छन्।


## एउटा अवलोकनयोग्य एजेन्ट निर्माण गर्दै

हामी यात्रा उपकरणहरू परिभाषित गर्छौं र एजेन्ट कललाई समयसँगै जडान गर्छौं ताकि हामी ढिलोपनलाई अनुगमन गर्न सकौँ। उत्पादनमा तपाईँले OpenTelemetry वा त्यस्तै ट्रेसिङ ब्याकएन्डसँग एकीकृत गर्नुहुनेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन ढाँचा

सामान्य उत्पादन ढाँचामा दोस्रो एजेन्टलाई **मूल्याङ्कनकर्ता** को रूपमा प्रयोग गरिन्छ। मूल्याङ्कनकर्ताले प्राथमिक एजेन्टको जवाफलाई पूर्वनिर्धारित मापदण्डहरू जस्तै पूर्णता, शुद्धता, र उपयोगिताको आधारमा मूल्याङ्कन गर्छ।

यसले सक्षम पार्छ:
- प्रयोगकर्ताहरूलाई जवाफहरू पुग्नुअघि स्वचालित गुणस्तर ढोकाहरू
- प्रॉम्प्ट वा मोडेल परिवर्तन हुँदा पुनरावृत्ति पत्ता लगाउने
- समयसँगै एजेन्ट प्रदर्शनको निरन्तर अनुगमन


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत व्यवस्थापन रणनीतिहरू

उत्पादन एआई एजेन्टहरूको लागि लागत नियन्त्रण महत्वपूर्ण छ। यहाँ मुख्य रणनीतिहरू छन्:

| रणनीति | विवरण |
|---|---|
| **प्रॉम्प्ट अप्टिमाइजेशन** | प्रणाली निर्देशनहरू संक्षिप्त राख्नुहोस्। इनपुट टोकनहरू घटाउन दोहोरिएको सन्दर्भ हटाउनुहोस्। |
| **मोडेल छनोट** | वर्गीकरण वा निकासी जस्ता सरल कार्यहरूका लागि साना र सस्तो मोडेलहरू (जस्तै GPT-4o-mini) प्रयोग गर्नुहोस्, र जटिल तर्कका लागि ठूलो मोडेलहरू जोगाउनुहोस्। |
| **क्याचिङ** | उपकरणका परिणामहरू र प्रायः सोधिने प्रश्नहरू क्याच गरेर दोहोरिएको API कलहरूबाट बच्नुहोस्। |
| **टोकन बजेटहरू** | अप्रत्याशित रूपले लामो प्रतिक्रियाहरू रोक्न `max_tokens` सीमा सेट गर्नुहोस्। |
| **ब्याचिंग** | सम्भव भएमा धेरै प्रयोगकर्ता प्रश्नहरूलाई एकल API कलमा समूहीकृत गर्नुहोस्। |

व्यवहारमा, एक तह गरिएको दृष्टिकोण राम्रो काम गर्छ: सिधा अनुरोधहरूलाई छिटो र सस्तो मोडेलमा मार्गनिर्देशन गर्नुहोस् र जटिल प्रश्नहरूलाई मात्र अधिक सक्षम मोडेलमा बढाउनुहोस्।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. एजेन्ट अन्तरक्रियामा **अवलोकनशीलता थप्ने** समय र लगिङ्ग मार्फत, जसले ट्रेसिङ र अनुगमनको आधार तयार पार्छ।
2. एजेन्ट प्रतिक्रियाहरू **स्वचालित रूपमा मूल्यांकन गर्ने** एउटा मूल्यांकनकर्ता एजेन्ट प्रयोग गरी जुन पूर्णता, शुद्धता, र सहयोगीपनको स्कोर गर्छ।
3. **लागतहरू व्यवस्थापन गर्ने** प्रॉम्प्ट अप्टिमाइजेशन, मोडेल चयन, क्याचिङ, र टोकन बजेटहरूको माध्यमबाट।

यी उत्पादन ढाँचाहरूले तपाईंका AI एजेन्टहरूलाई भरपर्दो, मापन योग्य, र ठूलो परिमाणमा लागत-प्रभावी बनाउन मद्दत गर्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
